# Sanjavan Ghodasara

**Cluster 2 Stacking Model**

**Group 2** | Cluster 2: 424 companies, 113 bankrupt (26.65% rate)

**No-SMOTE version.** Threshold lowered to **0.28** from 0.40. The team generalization sparsity check showed room in the budget, and since the spec scores only accuracy, a more aggressive (lower) threshold captures more true bankruptcies at the cost of higher sparsity which is an acceptable tradeoff.

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.feature_selection import mutual_info_classif

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
CLUSTER_DIR = os.path.join(DATA_DIR, 'clusters')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
SUBGROUP_MODEL_DIR = os.path.join(MODEL_DIR, 'subgroup_models')
os.makedirs(SUBGROUP_MODEL_DIR, exist_ok=True)

In [45]:
def custom_accuracy(y_true, y_pred):
    tt = ((y_true == 1) & (y_pred == 1)).sum()
    tf = ((y_true == 1) & (y_pred == 0)).sum()
    if tt + tf == 0:
        return 0.0
    return tt / (tt + tf)

def evaluate_model(y_true, y_pred, label=''):
    acc = custom_accuracy(y_true, y_pred)
    rate = y_pred.mean()
    cm = confusion_matrix(y_true, y_pred)
    tt = int(((y_true == 1) & (y_pred == 1)).sum())
    tf = int(((y_true == 1) & (y_pred == 0)).sum())
    n_pred = int(y_pred.sum())
    print(f'--- {label} ---')
    print(f'TT (bankrupt predicted bankrupt):     {tt}')
    print(f'TF (bankrupt predicted non-bankrupt): {tf}')
    print(f'TT + TF = {tt + tf} (total bankrupt)')
    print(f'N predicted bankrupt:                 {n_pred}')
    print(f'Custom Accuracy (TT/(TT+TF)): {acc:.4f}')
    print(f'Sparsity (predicted bankrupt %): {rate:.4f}')
    print(f'Confusion Matrix:\n{cm}')
    print(f'Classification Report:\n{classification_report(y_true, y_pred, zero_division=0)}')
    return acc, rate

---
## Load & Inspect Data

In [46]:
df2 = pd.read_csv(os.path.join(CLUSTER_DIR, 'cluster_2.csv'))
y2 = df2['Bankrupt?'].values
X2 = df2.drop(columns=['Bankrupt?'])
feature_names = X2.columns.tolist()

print(f'Cluster 2: {X2.shape[0]} samples, {X2.shape[1]} features')
print(f'Bankrupt: {int(y2.sum())} ({y2.mean():.4f})')
print(f'Non-bankrupt: {int((y2 == 0).sum())}')

Cluster 2: 424 samples, 95 features
Bankrupt: 113 (0.2665)
Non-bankrupt: 311


## Config

- **Base:** RF + GradientBoosting + LogisticRegression (all `class_weight='balanced'`)
- **Meta:** `LogisticRegression(C=2.0)`
- **SMOTE:** NONE — the balanced `class_weight` on RF and LR is sufficient
- **N = 5** features
- **Threshold = 0.28** which was lowered from 0.40 because the team generalization sparsity budget has room and the spec only scores accuracy; a lower threshold captures more true bankruptcies, improving custom accuracy from 71.68% → 84.96% (train) and 44.25% → 69.03% (CV)

### Why no SMOTE
An ablation sweep at the same N=5 features and same base stack showed that without
SMOTE, the model achieves a strictly higher training TT (112 vs 109) at a slightly
higher but acceptable sparsity. The class_weight='balanced' setting already pulls
the loss toward minority-class examples; additional synthetic oversampling was
diluting the signal.

## Feature Selection

In [47]:
mi_scores = mutual_info_classif(X2.values, y2, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=feature_names).sort_values(ascending=False)

print('Top 15 features by Mutual Information:')
for i, (feat, score) in enumerate(mi_series.head(15).items()):
    print(f'  {i+1:2d}. {feat[:55]:55s}  MI = {score:.4f}')

corr_matrix = X2.corr().abs()
CORR_THRESH = 0.85
selected = []
for feat in mi_series.index:
    if not selected or not any(corr_matrix.loc[feat, s] > CORR_THRESH for s in selected):
        selected.append(feat)
    if len(selected) >= 10:
        break

N_FEATURES = 5
top_features = selected[:N_FEATURES]
X2_sel = df2[top_features].values

print(f'\nSelected {N_FEATURES} diverse features (pairwise |r| < {CORR_THRESH}):')
for i, feat in enumerate(top_features):
    print(f'  {i+1}. {feat[:55]:55s}  MI = {mi_series[feat]:.4f}')

Top 15 features by Mutual Information:
   1. Borrowing dependency                                     MI = 0.0918
   2. Total debt/Total net worth                               MI = 0.0857
   3. Debt ratio %                                             MI = 0.0827
   4. Net worth/Assets                                         MI = 0.0825
   5. Equity to Liability                                      MI = 0.0812
   6. Liability to Equity                                      MI = 0.0798
   7. Net profit before tax/Paid-in capital                    MI = 0.0702
   8. Persistent EPS in the Last Four Seasons                  MI = 0.0700
   9. Net Value Per Share (B)                                  MI = 0.0654
  10. Per Share Net profit before tax (Yuan ¥)                 MI = 0.0572
  11. Net Income to Stockholder's Equity                       MI = 0.0537
  12. Net Value Per Share (A)                                  MI = 0.0536
  13. Total income/Total expense                             

## Stacking Model Definition

In [48]:
BEST_THRESH = 0.28

print(f'Using {N_FEATURES} features, NO SMOTE, threshold={BEST_THRESH}')

base_estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
    ('gb', GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE)),
    ('lr', LogisticRegression(
        class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
]

meta_learner = LogisticRegression(
    C=2.0, max_iter=1000, random_state=RANDOM_STATE)

print('Stacking classifier: RF + GB + LR base + LR(C=2.0) meta, no SMOTE.')

Using 5 features, NO SMOTE, threshold=0.28
Stacking classifier: RF + GB + LR base + LR(C=2.0) meta, no SMOTE.


## Individual Base-Model Diagnostics

In [49]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def base_cv_predict(model_factory, X_raw, y):
    preds = np.zeros(len(y), dtype=int)
    for train_idx, val_idx in outer_cv.split(X_raw, y):
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_raw[train_idx])
        X_val = sc.transform(X_raw[val_idx])
        m = model_factory()
        m.fit(X_tr, y[train_idx])
        preds[val_idx] = m.predict(X_val)
    return preds

rf_preds_cv = base_cv_predict(
    lambda: RandomForestClassifier(
        n_estimators=200, max_depth=6, min_samples_leaf=5,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1),
    X2_sel, y2)
evaluate_model(y2, rf_preds_cv, 'Base: Random Forest (CV preds, thresh=0.5)')

gb_preds_cv = base_cv_predict(
    lambda: GradientBoostingClassifier(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        min_samples_leaf=10, random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, gb_preds_cv, 'Base: Gradient Boosting (CV preds, thresh=0.5)')

lr_preds_cv = base_cv_predict(
    lambda: LogisticRegression(class_weight='balanced', C=1.0, max_iter=2000,
                               random_state=RANDOM_STATE),
    X2_sel, y2)
evaluate_model(y2, lr_preds_cv, 'Base: Logistic Regression (CV preds, thresh=0.5)')

--- Base: Random Forest (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     67
TF (bankrupt predicted non-bankrupt): 46
TT + TF = 113 (total bankrupt)
N predicted bankrupt:                 143
Custom Accuracy (TT/(TT+TF)): 0.5929
Sparsity (predicted bankrupt %): 0.3373
Confusion Matrix:
[[235  76]
 [ 46  67]]
Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.76      0.79       311
           1       0.47      0.59      0.52       113

    accuracy                           0.71       424
   macro avg       0.65      0.67      0.66       424
weighted avg       0.74      0.71      0.72       424

--- Base: Gradient Boosting (CV preds, thresh=0.5) ---
TT (bankrupt predicted bankrupt):     44
TF (bankrupt predicted non-bankrupt): 69
TT + TF = 113 (total bankrupt)
N predicted bankrupt:                 75
Custom Accuracy (TT/(TT+TF)): 0.3894
Sparsity (predicted bankrupt %): 0.1769
Confusion Matrix:
[[280  31]
 [ 69  44]]

(np.float64(0.6106194690265486), np.float64(0.3490566037735849))

## Cross-Validation (Stacked Model)

5-fold stratified CV. Per-fold StandardScaler only. No SMOTE.

In [50]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y2_cv_probas = np.zeros(len(y2))

for fold, (train_idx, val_idx) in enumerate(cv.split(X2_sel, y2)):
    sc_fold = StandardScaler()
    X_train_cv = sc_fold.fit_transform(X2_sel[train_idx])
    X_val_cv = sc_fold.transform(X2_sel[val_idx])
    y_train_cv = y2[train_idx]

    stacker_fold = StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(
                n_estimators=200, max_depth=6, min_samples_leaf=5,
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
            ('gb', GradientBoostingClassifier(
                n_estimators=150, max_depth=3, learning_rate=0.05,
                min_samples_leaf=10, random_state=RANDOM_STATE)),
            ('lr', LogisticRegression(
                class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
        ],
        final_estimator=LogisticRegression(
            C=2.0, max_iter=1000, random_state=RANDOM_STATE),
        cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
        stack_method='predict_proba',
        n_jobs=-1
    )
    stacker_fold.fit(X_train_cv, y_train_cv)
    y2_cv_probas[val_idx] = stacker_fold.predict_proba(X_val_cv)[:, 1]
    print(f'Fold {fold+1}: val positives={int(y2[val_idx].sum())}, avg proba bankrupt={y2_cv_probas[val_idx].mean():.4f}')

Fold 1: val positives=22, avg proba bankrupt=0.2383
Fold 2: val positives=23, avg proba bankrupt=0.2390
Fold 3: val positives=23, avg proba bankrupt=0.2910
Fold 4: val positives=23, avg proba bankrupt=0.2842
Fold 5: val positives=22, avg proba bankrupt=0.2835


## Train Final Model & Threshold Sweep

In [51]:
scaler_final = StandardScaler()
X2_scaled = scaler_final.fit_transform(X2_sel)

stacker_final = StackingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=5,
            class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ('gb', GradientBoostingClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05,
            min_samples_leaf=10, random_state=RANDOM_STATE)),
        ('lr', LogisticRegression(
            class_weight='balanced', C=1.0, max_iter=2000, random_state=RANDOM_STATE)),
    ],
    final_estimator=LogisticRegression(
        C=2.0, max_iter=1000, random_state=RANDOM_STATE),
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE),
    stack_method='predict_proba',
    n_jobs=-1
)
stacker_final.fit(X2_scaled, y2)

y2_train_proba = stacker_final.predict_proba(X2_scaled)[:, 1]

print(f'--- Threshold Sweep (Training predictions) ---')
print(f'{"thresh":>7s} {"TR_TT":>5s} {"TR_TF":>5s} {"TR_acc":>7s} {"CV_TT":>5s} {"CV_TF":>5s} {"CV_acc":>7s} {"N_pred":>6s} {"sparsity":>8s}')
for t in np.arange(0.10, 0.501, 0.01):
    cv_p = (y2_cv_probas >= t).astype(int)
    tr_p = (y2_train_proba >= t).astype(int)
    cv_tt=int(((y2==1)&(cv_p==1)).sum()); cv_tf=int(((y2==1)&(cv_p==0)).sum())
    tr_tt=int(((y2==1)&(tr_p==1)).sum()); tr_tf=int(((y2==1)&(tr_p==0)).sum())
    cv_acc = cv_tt/(cv_tt+cv_tf) if (cv_tt+cv_tf)>0 else 0
    tr_acc = tr_tt/(tr_tt+tr_tf) if (tr_tt+tr_tf)>0 else 0
    if round(t*100) % 2 == 0 or round(t, 2) in [0.14, 0.15, 0.17]:
        print(f'  {t:5.2f}  {tr_tt:5d} {tr_tf:5d} {tr_acc:7.4f} {cv_tt:5d} {cv_tf:5d} {cv_acc:7.4f} {int(tr_p.sum()):6d} {tr_p.mean():8.4f}')

print(f'\nCommitted threshold: {BEST_THRESH:.2f}')

y2_cv_preds = (y2_cv_probas >= BEST_THRESH).astype(int)
y2_train_pred = (y2_train_proba >= BEST_THRESH).astype(int)

evaluate_model(y2, y2_cv_preds, f'Stacked: Cluster 2 — 5-Fold CV (thresh={BEST_THRESH:.2f})')
evaluate_model(y2, y2_train_pred, f'Stacked: Cluster 2 — Training on full cluster (thresh={BEST_THRESH:.2f})')

print('\n--- Individual Base Models (trained, full data, thresh=0.5) ---')
for name, est in stacker_final.named_estimators_.items():
    base_preds = est.predict(X2_scaled)
    evaluate_model(y2, base_preds, f'Base(trained): {name}')

--- Threshold Sweep (Training predictions) ---
 thresh TR_TT TR_TF  TR_acc CV_TT CV_TF  CV_acc N_pred sparsity
   0.10    112     1  0.9912   112     1  0.9912    337   0.7948
   0.12    112     1  0.9912   109     4  0.9646    305   0.7193
   0.14    112     1  0.9912   107     6  0.9469    278   0.6557
   0.15    111     2  0.9823   105     8  0.9292    267   0.6297
   0.16    111     2  0.9823   104     9  0.9204    257   0.6061
   0.17    111     2  0.9823    99    14  0.8761    251   0.5920
   0.18    109     4  0.9646    99    14  0.8761    241   0.5684
   0.20    108     5  0.9558    92    21  0.8142    225   0.5307
   0.22    105     8  0.9292    88    25  0.7788    205   0.4835
   0.24    102    11  0.9027    87    26  0.7699    184   0.4340
   0.26    100    13  0.8850    84    29  0.7434    166   0.3915
   0.28     96    17  0.8496    78    35  0.6903    151   0.3561
   0.30     96    17  0.8496    68    45  0.6018    146   0.3443
   0.32     93    20  0.8230    66    47  0.

In [52]:
model_info = {
    'model': stacker_final,
    'scaler': scaler_final,
    'features': top_features,
    'n_features': N_FEATURES,
    'threshold': BEST_THRESH,
    'cluster_id': 2,
    'model_type': 'stacking',
    'base_models': 'RF + GB + LR',
    'meta_model': 'LogisticRegression(C=2.0)',
    'smote_ratio': None,
    'threshold_selection': 'max training TT/(TT+TF), no SMOTE',
    'member': 'Sanjavan Ghodasara',
    'data_version': 'refreshed (424 rows, 26.65% bankrupt)',
}
joblib.dump(model_info, os.path.join(SUBGROUP_MODEL_DIR, 'cluster_2_model.joblib'))
print(f'Cluster 2 model saved. Features: {N_FEATURES}, Threshold: {BEST_THRESH}')
print(f'Saved keys: {sorted(model_info.keys())}')

Cluster 2 model saved. Features: 5, Threshold: 0.28
Saved keys: ['base_models', 'cluster_id', 'data_version', 'features', 'member', 'meta_model', 'model', 'model_type', 'n_features', 'scaler', 'smote_ratio', 'threshold', 'threshold_selection']


## Summary Cluster 2 (no-SMOTE version)

| Metric | CV (5-fold) | Training |
|---|---|---|
| TT (bankrupt predicted bankrupt) | **78** | **96** |
| TF (bankrupt predicted non-bankrupt) | **35** | **17** |
| TT + TF | 113 | 113 |
| Custom Accuracy (TT/(TT+TF)) | **69.03%** | **84.96%** |
| N predicted bankrupt | 171 | 151 |
| Sparsity (predicted bankrupt %) | 40.33% | 35.61% |
| Features (N) | 5 | 5 |
| Threshold | 0.28 | 0.28 |
| Feature Score ((50-N)/50) | 0.90 | 0.90 |

**Rank Score (local):** 0.3(0.8496) + 0.4(0.6903) + 0.3(0.90) = **0.8010**

**Top 5 Features:**
1. Borrowing dependency
2. Total debt/Total net worth
3. Debt ratio %
4. Equity to Liability
5. Net profit before tax/Paid-in capital

### Why t=0.28 instead of the prior t=0.40

The team generalization sparsity check showed room in the budget. Since the spec only scores custom accuracy (TT/(TT+TF)), lowering the threshold from 0.40 to 0.28 increases training accuracy (71.68% → 84.96%) and CV accuracy (44.25% → 69.03%) at the cost of higher sparsity (22.88% → 35.61%) which is an acceptable tradeoff given the available team sparsity budget.

| Threshold | Train TT/TF | Train acc | Train sparsity |
|---|---|---|---|
| 0.14 (original) | 112 / 1 | 99.12% | 65.57% |
| 0.40 (prior) | 81 / 32 | 71.68% | 22.88% |
| **0.28 (now)** | **96 / 17** | **84.96%** | **35.61%** |

### Saved Artifacts
- `models/subgroup_models/cluster_2_model.joblib` — fitted stack + scaler +
  feature list + threshold=0.28